# Chapter 27: Bigger Data and Faster Code

The programs from Chapter 27 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## Smaller Types, Less Memory

`memory.py`

In [1]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")
before = orders.memory_usage(deep=True).sum()

smaller = orders.astype(
    {
        "region": "category",
        "channel": "category",
        "category": "category",
        "product": "category",
        "quantity": "int8",
        "discount": "float32",
    }
)
after = smaller.memory_usage(deep=True).sum()
print(
    f"before {before / 1e6:.2f} MB, after {after / 1e6:.2f} MB "
    f"({after / before:.0%} of the original)"
)
print(smaller.dtypes[["region", "quantity", "discount"]])

before 2.41 MB, after 1.08 MB (45% of the original)
region      category
quantity        int8
discount     float32
dtype: object


## Reading in Chunks

`chunks.py`

In [2]:
import pandas as pd

totals = {}
for chunk in pd.read_csv("../data/sessions.csv", chunksize=5_000):
    part = chunk.groupby("variant")["revenue"].sum()
    for variant, revenue in part.items():
        totals[variant] = totals.get(variant, 0) + revenue
    print(f"read {len(chunk):,} rows; running total A {totals['A']:,.2f}")

whole = pd.read_csv("../data/sessions.csv").groupby("variant")["revenue"]
print(
    "same as reading it all at once:",
    all(abs(totals[v] - s) < 0.01 for v, s in whole.sum().items()),
)

read 5,000 rows; running total A 6,800.99
read 5,000 rows; running total A 13,741.50
read 5,000 rows; running total A 20,230.35
read 5,000 rows; running total A 27,372.98
same as reading it all at once: True


## Readable Pipelines

`method_chain.py`

In [3]:
import pandas as pd


def add_revenue(df):
    return df.assign(
        revenue=df["quantity"] * df["unit_price"] * (1 - df["discount"])
    )


summary = (
    pd.read_csv("../data/orders.csv", parse_dates=["order_date"])
    .pipe(add_revenue)
    .query("channel != 'marketplace'")
    .groupby(["region", "channel"], as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
    .head(4)
    .round(0)
)
print(summary.to_string(index=False))

region channel  revenue
 North     web  55847.0
  East     web  52391.0
 South     web  43196.0
  West     web  37230.0
